In [2]:
import pandas as pd
import numpy as np
import re
import torch

from sklearn.neighbors import BallTree
from torch_geometric.data import Data

# -------------------------------------------------
# Load Grid Coordinates
# -------------------------------------------------

rain_df = pd.read_csv("grid_with_rainfall.csv")

def parse_centroid(c):
    nums = re.findall(r"[-+]?\d*\.\d+|\d+", c)
    lat, lon = map(float, nums)
    return lat, lon

coords = np.array([parse_centroid(c) for c in rain_df["centroid"]])

print("Grid shape:", coords.shape)

# -------------------------------------------------
# Convert Coordinates to Radians
# Required for Haversine Distance
# -------------------------------------------------

coords_rad = np.radians(coords)

# -------------------------------------------------
# Build Static k-NN Graph using Haversine Distance
# -------------------------------------------------

k = 8

tree = BallTree(coords_rad, metric='haversine')

distances, indices = tree.query(coords_rad, k=k+1)

# -------------------------------------------------
# Create Edge Index
# -------------------------------------------------

edge_index = []

for i, nbrs in enumerate(indices):

    # Skip self-loop (first neighbor is itself)
    for j in nbrs[1:]:
        edge_index.append([i, j])

edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

# -------------------------------------------------
# Save Graph
# -------------------------------------------------

data = Data(edge_index=edge_index)

torch.save(data, "static_spatial_graph.pt")

print("Static graph saved successfully")

# -------------------------------------------------
# Save Static Node Features
# -------------------------------------------------

temp_df = pd.read_csv("grid_with_temperature.csv")

x = torch.tensor(
    np.column_stack([
        rain_df["rainfall"].values,
        temp_df["temperature"].values
    ]),
    dtype=torch.float
)

# -------------------------------------------------
# Save Features for Each Year
# -------------------------------------------------

years = range(2000, 2024)

for y in years:
    torch.save(x, f"node_features_{y}.pt")

print("Static node features saved for all years")

# -------------------------------------------------
# Graph Information
# -------------------------------------------------

print("\nGraph Construction Summary")
print("--------------------------")
print(f"Number of Nodes : {coords.shape[0]}")
print(f"k-nearest Neighbors : {k}")
print("Distance Metric : Haversine")
print(f"Number of Edges : {edge_index.shape[1]}")


Grid shape: (5040, 2)
Static graph saved successfully
Static node features saved for all years

Graph Construction Summary
--------------------------
Number of Nodes : 5040
k-nearest Neighbors : 8
Distance Metric : Haversine
Number of Edges : 40320
